# Inspect a spatial SAE activation

Choose an image sample and SAE feature. This notebook displays the 16 × 16 feature map as colored pixel cells over the generated image and can overlay the corresponding C2C box.

The activation, image, SAE checkpoint, and optional C2C labels are read from the companion `debiasing` workspace. Run on a CUDA GPU. The default paths assume this repository sits at `/n/fs/goose/sae-demo-notebooks`, beside the `debiasing` files at `/n/fs/goose`. Edit `DEBIASING_ROOT` if your layout differs.

In [ ]:
from pathlib import Path
import sys
import csv
import json
import numpy as np
import torch
import ipywidgets as widgets
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from IPython.display import display, clear_output

NOTEBOOK_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
                     if p.name == "sae-demo-notebooks" and (p / "sae-activation").exists())
DEBIASING_ROOT = NOTEBOOK_ROOT.parent  # Edit this if debiasing is elsewhere.
BLOCK = "unet.down_blocks.2.attentions.1"
CHECKPOINT = DEBIASING_ROOT / "sdxl-unbox/checkpoints" / f"{BLOCK}_k10_hidden5120_auxk256_bs4096_lr0.0001/final"
RESIDUAL_DIR = DEBIASING_ROOT / "sae-training/residuals"
MANIFEST = DEBIASING_ROOT / "backprop/manifest.csv"
BOX_TARGETS = DEBIASING_ROOT / "sae-training/c2c_box_targets.csv"

sys.path.insert(0, str(DEBIASING_ROOT / "sdxl-unbox"))
from SAE import SparseAutoencoder

if not torch.cuda.is_available():
    raise RuntimeError("Run this notebook on a CUDA GPU node.")


## Select a sample and feature

The sample ID must have a saved residual and generated image in the companion workspace. The default feature is just an example; a high activation does not by itself prove that the feature means “flower.”

In [ ]:
sample_id = widgets.Text(value="east_asian_woman_seed000", description="Sample:", layout=widgets.Layout(width="450px"))
feature_id = widgets.IntText(value=2810, description="Feature:")
concept = widgets.Dropdown(options=["flower", "kimono"], value="flower", description="C2C box:")
alpha = widgets.FloatSlider(value=0.5, min=0.1, max=1.0, step=0.1, description="Overlay:")
button = widgets.Button(description="Show activation", button_style="primary")
output = widgets.Output()

with MANIFEST.open(newline="") as f:
    manifest = {row["sample_id"]: row for row in csv.DictReader(f)}


In [ ]:
def show_activation(_):
    with output:
        clear_output(wait=True)
        sid, fid = sample_id.value.strip(), int(feature_id.value)
        try:
            row = manifest[sid]
            residual_path = RESIDUAL_DIR / f"{sid}.npy"
            image_path = DEBIASING_ROOT / "backprop" / row["image"]
            if not residual_path.exists() or not image_path.exists():
                raise FileNotFoundError(f"Need both {residual_path} and {image_path}")
            device = torch.device("cuda")
            sae = SparseAutoencoder.load_from_disk(str(CHECKPOINT)).float().to(device).eval()
            residual = torch.from_numpy(np.load(residual_path, allow_pickle=False).astype(np.float32)).to(device)
            h, w, d = residual.shape
            if fid < 0 or fid >= sae.n_dirs:
                raise ValueError(f"Feature ID must be between 0 and {sae.n_dirs - 1}")
            with torch.inference_mode():
                activation = sae.encode(residual.reshape(h*w, d))[:, fid].reshape(h, w).cpu().numpy()
            image = plt.imread(image_path)
            fig, ax = plt.subplots(figsize=(8, 8))
            ax.imshow(image)
            shown = np.ma.masked_where(activation <= 0, activation)
            heat = ax.imshow(shown, cmap="jet", alpha=alpha.value, interpolation="nearest",
                             extent=(0, image.shape[1], image.shape[0], 0), origin="upper")
            box_row = next((r for r in csv.DictReader(BOX_TARGETS.open(newline=""))
                            if r["sample_id"] == sid and r["concept"] == concept.value), None)
            if box_row:
                boxes = json.loads(box_row["boxes_xyxy_normalized"])
                for x1, y1, x2, y2 in boxes:
                    ax.add_patch(Rectangle((x1*image.shape[1], y1*image.shape[0]),
                                           (x2-x1)*image.shape[1], (y2-y1)*image.shape[0],
                                           fill=False, edgecolor="white", linewidth=2))
            ax.set_title(f"{sid} · SAE feature {fid} · {concept.value} C2C box")
            ax.axis("off")
            fig.colorbar(heat, ax=ax, fraction=0.046, pad=0.04, label="SAE activation")
            plt.show()
            print(f"max activation={activation.max():.4f}; active cells={(activation > 0).sum()}/{activation.size}")
            del sae, residual
            torch.cuda.empty_cache()
        except Exception as exc:
            print(f"Could not display activation: {type(exc).__name__}: {exc}")

button.on_click(show_activation)
display(sample_id, feature_id, concept, alpha, button, output)


The overlay uses a nearest-neighbor grid so each activation corresponds to one spatial SAE cell. White rectangles come from the C2C working labels and may be incorrect; inspect the source image before treating them as ground truth.